# Clase 4 asincrónica

Este notebook contiene el código de las slides de la clase 4, en el mismo orden. El número de slide figura en cada título.

**Antes de empezar:** guardá una copia con otro nombre (*Save as*) y trabajá sobre la copia. Así evitás conflictos de `git` al hacer `pull` más adelante.

En las celdas tituladas **Pausa**, escribí tu predicción *antes* de ejecutar el código.

## 1. Pandas (continuación)

### Un DataFrame de ejemplo (slide 9)

In [1]:
import numpy as np
import pandas as pd

In [6]:
np.random.seed(73)      # semilla: todos obtenemos los mismos números
df = pd.DataFrame({"media": [0]*1000 + [1]*1000 + [2]*100})
df


,media
0,0
1,0
2,0
3,0
4,0
...,...
2095,2
2096,2
2097,2
2098,2


In [7]:
def nueva_columna(tabla):
    return tabla["media"] + np.random.randn(tabla.shape[0])

df2 = df.assign(x=nueva_columna)   # assign devuelve un DataFrame nuevo
df

,media
0,0
1,0
2,0
3,0
4,0
...,...
2095,2
2096,2
2097,2
2098,2


### Funciones anónimas: `lambda` (slide 10)

La misma columna, definida con una función anónima. Fijamos otra vez la semilla para obtener los mismos números.

In [14]:
np.random.seed(73)
df = pd.DataFrame({"media": [0]*1000 + [1]*1000 + [2]*100})
df = df.assign(x=lambda d: d["media"] + np.random.randn(d.shape[0]))
df.head()

,media,x
0,0,0.576813
1,0,2.131109
2,0,2.440220
3,0,0.263327
4,0,-1.496121


### Transformaciones sobre grupos: estandarizar (slide 11)

In [ ]:
df["t"] = df.groupby("media")["x"].transform(
    lambda s: (s - s.mean()) / s.std())

Verificamos: dentro de cada grupo, `t` debe tener media 0 y desvío 1.

In [ ]:
df.groupby("media")[["x", "t"]].agg(["mean", "std"]).round(2)

In [ ]:
df.shape

### Transformaciones sobre grupos: imputar faltantes (slide 12)

In [ ]:
sal = pd.DataFrame({
    "provincia": ["A", "A", "A", "B", "B", "B"],
    "salario":   [100, np.nan, 140, 200, 220, np.nan]})

sal["salario_imp"] = sal.groupby("provincia")["salario"].transform(
    lambda s: s.fillna(s.mean()))
sal

### Filtrado por grupos (slide 13)

Se conservan los grupos para los que la función devuelve `True`.

In [ ]:
grandes = df.groupby("media").filter(lambda g: len(g) >= 500)
grandes.shape

In [ ]:
df.groupby("media").filter(lambda g: len(g) < 500).shape

### Pausa: ¿cuántas filas devuelve cada línea? (slide 14)

Escribí acá tu predicción **antes** de ejecutar las celdas de abajo:

- Filas del resultado (1):
- Filas del resultado (2), y sus valores:
- Filas del resultado (3):

In [ ]:
p = pd.DataFrame({"g": ["a", "a", "a", "b", "b"],
                  "v": [ 1,   2,   3,   10,  20]})

In [ ]:
p.groupby("g")["v"].mean()                             # (1)

In [ ]:
p.groupby("g")["v"].transform("mean")                  # (2)

In [ ]:
p.groupby("g").filter(lambda d: d["v"].mean() > 5)     # (3)

### De ancho a largo: `melt` (slide 16)

In [ ]:
df3 = pd.DataFrame({"first": ["John", "Mary"], "last": ["Doe", "Bo"],
                    "height": [5.5, 6.0], "weight": [130, 150]})
df3

In [ ]:
df3.melt(id_vars=["first", "last"])

### `melt` con un panel de datos (slide 17)

In [ ]:
ancho = pd.DataFrame({"provincia": ["Chaco", "Salta"],
    "2022": [10.5, 12.0], "2023": [11.0, 12.4], "2024": [11.8, 13.1]})

largo = ancho.melt(id_vars="provincia",
                   var_name="anio", value_name="valor")
largo

El camino inverso, de largo a ancho:

In [ ]:
largo.pivot(index="provincia", columns="anio", values="valor")

### Pregunta 1 del cuestionario (slide 18)

Respondé primero sin ejecutar nada. Después, comprobá tu respuesta agregando una celda con el `melt`.

In [ ]:
df_p1 = pd.DataFrame({
    "Nombre": ["Mabel", "Juan", "Héctor"],
    "Apellido": ["García", "Pérez", "Rossi"],
    "Edad": [40, 50, 60],
    "Dirección": ["San Martín 900", "Belgrano 1770", "Sarmiento 1888"],
    "Ciudad": ["La Plata", "Rosario", "Santa Rosa"]})
df_p1

## 2. Visualización con `seaborn`

### Los datos del ejemplo: `tips` (slides 20 y 21)

In [ ]:
import seaborn as sns

In [ ]:
propinas_df = sns.load_dataset("tips")
propinas_df.shape

In [ ]:
propinas_df.head(3)

### Un gráfico de dispersión: `relplot` (slide 22)

In [ ]:
sns.set_theme()                       # tema por defecto de seaborn
sns.relplot(data=propinas_df, x="total_bill", y="tip", hue="sex");

### Paneles: `row` y `col` (slide 23)

In [ ]:
sns.relplot(data=propinas_df, x="total_bill", y="tip", hue="sex",
            row="time", col="smoker");

### Tema, estilo y paleta de colores (slide 24)

In [ ]:
sns.set_theme(style="whitegrid", palette="pastel")
sns.relplot(data=propinas_df, x="total_bill", y="tip", hue="sex");

In [ ]:
sns.set_theme()     # volvemos al tema por defecto

### Una variable categórica: `boxplot` (slide 25)

In [ ]:
sns.boxplot(data=propinas_df, x="day", y="total_bill");

### Graficar una variable que todavía no existe (slide 26)

In [ ]:
sns.boxplot(
    data=propinas_df.assign(fraccion=lambda d: d["tip"] / d["total_bill"]),
    x="day", y="fraccion");

### Pregunta 2 del cuestionario (slide 27)

Escribí en la celda de abajo la línea que genera el boxplot pedido.

## 3. Expresiones regulares

### Un primer patrón: `compile` y `search` (slides 30 y 31)

In [ ]:
import re

In [ ]:
frase = "¿Dónde va Vicente? Donde va la gente."

regex = re.compile(r"ente")     # objeto de tipo re.Pattern
regex.search(frase)             # busca la primera coincidencia

In [ ]:
frase[13:17]

### Todas las coincidencias: `findall` y `finditer` (slide 32)

In [ ]:
regex.findall(frase)            # lista de strings

In [ ]:
for coincidencia in regex.finditer(frase):     # iterador de re.Match
    print(coincidencia)

In [ ]:
m = regex.search(frase)
m.group(), m.start(), m.end()

### Conjuntos de caracteres (slides 33 y 34)

In [ ]:
re.findall(r"[Hhbcs]ola", "Hola hola bola cola sola mola")

In [ ]:
re.findall(r"[^bc]ola", "Hola hola bola cola sola mola")

In [ ]:
re.findall(r"Donde", frase)

In [ ]:
re.findall(r"D[oó]nde", frase)

In [ ]:
re.findall(r"[^c]ente", frase)

In [ ]:
re.findall(r"\w+", "tasa_2024: 3,5%")

### Caracteres especiales y escape (slide 35)

In [ ]:
re.findall(r"1.5",  "1.5 y 105")     # el punto es comodín

In [ ]:
re.findall(r"1\.5", "1.5 y 105")     # punto literal

### Cuantificadores (slides 36 y 37)

In [ ]:
frase2 = frase + " ¿Por qué? Porque es un ente."

re.findall(r"\w+ente", frase2)     # una o más letras antes

In [ ]:
re.findall(r"\w*ente", frase2)     # cero o más letras antes

In [ ]:
re.findall(r"\w*ente", "atentamente")

### Alternativas: la barra vertical (slide 38)

In [ ]:
re.findall(r"\w*ente|D[oó]nde", frase2)

### Pausa: ¿qué devuelve cada línea? (slide 39)

Escribí acá tu predicción **antes** de ejecutar las celdas de abajo:

- Lista (1):
- Lista (2):
- Patrón para quedarse solo con los años:

In [ ]:
texto = "En 2023 la inflación fue 211,4% y en 2024, 117,8%"

In [ ]:
re.findall(r"\d+", texto)             # (1)

In [ ]:
re.findall(r"\d+,\d+%", texto)        # (2)

### Buscar: `search`, `match` y `fullmatch` (slide 41)

In [ ]:
for t in ["2024", "2024 pesos", "año 2024"]:
    print(t, "|",
          bool(re.search(r"\d+", t)),
          bool(re.match(r"\d+", t)),
          bool(re.fullmatch(r"\d+", t)))

### Listar, dividir y reemplazar (slide 42)

In [ ]:
re.split(r"\s*[;,]\s*", "soja; maíz , trigo,girasol")

In [ ]:
re.sub(r"\s+", " ", "Av.   7   N°  776")

### Extraer partes de una coincidencia: grupos (slide 43)

In [ ]:
regex = re.compile(r"(.*)\.(txt|csv|xls|doc)")
m = regex.match("archivo_imaginario.xls")

print(m.group(1))
print(m.group(2))
print(m.group(0))
print(m.groups())

### ¿El patrón hace lo que queremos? Hay que probarlo (slide 44)

In [ ]:
casos = ["datos.csv", "tabla.v2.xls", "foto.png", "informe.xlsx", "notas.docx"]
for caso in casos:
    con_match = regex.match(caso)
    con_fullmatch = regex.fullmatch(caso)
    print(caso, "|",
          con_match.group(0) if con_match else None, "|",
          con_fullmatch.group(0) if con_fullmatch else None)

### Los paréntesis cambian lo que devuelve `findall` (slide 45)

In [ ]:
re.findall(r"\d+(,\d+)?%", texto)

In [ ]:
re.findall(r"\d+(?:,\d+)?%", texto)

### Aplicación: darle estructura al *Martín Fierro* (slides 47 a 51)

Queremos una lista anidada: `versos[i][j][k]` es el verso `k` de la estrofa `j` de la parte `i`.

**Paso 1.** Leer el archivo y separar las partes.

In [ ]:
with open("archivos/martin_fierro.txt", encoding="utf8") as archivo:
    martin_fierro = archivo.read()

print(martin_fierro[:600])

In [ ]:
partes = re.split(r"[IVX]+ - .*\n", martin_fierro)
len(partes)

In [ ]:
encabezado = partes.pop(0)     # pop quita el elemento y lo devuelve
len(partes)

**Pasos 2 y 3.** Estrofas y versos.

In [ ]:
estrofas = []
for parte in partes:
    lista = re.split(r"\n\d+\n", parte)    # salto, número, salto
    lista.pop(0)                           # string vacío inicial
    estrofas.append(lista)

versos = []
for parte in estrofas:
    lista = []
    for estrofa in parte:
        lista.append(estrofa.strip().split("\n"))
    versos.append(lista)

versos[0][0]

**Paso 4.** Verificar el resultado: ¿cuántos versos tiene cada estrofa?

In [ ]:
largos = []
for parte in versos:
    for estrofa in parte:
        largos.append(len(estrofa))

pd.Series(largos).value_counts().sort_index()

In [ ]:
versos[-1][-1][:14]     # la última estrofa trae el pie del archivo

Cortamos el texto antes del pie de Project Gutenberg y repetimos los pasos 1 a 3.

In [ ]:
texto_mf = martin_fierro.split("End of Project Gutenberg")[0]

partes = re.split(r"[IVX]+ - .*\n", texto_mf)
partes.pop(0)

estrofas = []
for parte in partes:
    lista = re.split(r"\n\d+\n", parte)
    lista.pop(0)
    estrofas.append(lista)

versos = []
for parte in estrofas:
    lista = []
    for estrofa in parte:
        lista.append(estrofa.strip().split("\n"))
    versos.append(lista)

largos = []
for parte in versos:
    for estrofa in parte:
        largos.append(len(estrofa))

print(len(versos), "partes,", len(largos), "estrofas,", sum(largos), "versos")
pd.Series(largos).value_counts().sort_index()

**Buscar en el texto estructurado** (slide 51): primera y última palabra de cada verso.

In [ ]:
for verso in versos[0][0]:
    print(re.findall(r"^\w+|\w+$", verso))

In [ ]:
for verso in versos[0][0]:
    print(re.findall(r"^\w+|\w+[.,;:]?$", verso))

¿En qué proporción de los versos de todo el poema encuentra las dos palabras cada patrón?

In [ ]:
for patron in [r"^\w+|\w+$", r"^\w+|\w+[.,;:]?$"]:
    total = 0
    completos = 0
    for parte in versos:
        for estrofa in parte:
            for verso in estrofa:
                total += 1
                if len(re.findall(patron, verso)) == 2:
                    completos += 1
    print(patron, round(100 * completos / total), "%")

### Pregunta 3 del cuestionario (slide 52)

Para cada expresión, armá una lista con al menos tres textos que deben coincidir y tres que no, y comprobá el resultado.

In [ ]:
casos = ["...", "...", "..."]          # completar
for caso in casos:
    print(caso, bool(re.search(r"...", caso)))

## 4. Widgets en Jupyter

Estas celdas tienen que ejecutarse en un notebook (Jupyter, VS Code o Colab).

### Crear un widget y leer su valor (slide 55)

In [ ]:
import ipywidgets as widgets

In [ ]:
w = widgets.IntSlider(min=0, max=10)
display(w)

In [ ]:
w.value     # mové la barra y volvé a ejecutar esta celda

In [ ]:
w2 = widgets.Textarea()
display(w2)

In [ ]:
w2.value

### Reactividad: `interact` (slide 56)

In [ ]:
from ipywidgets import interact

def f(x):
    print("El doble de", x, "es", 2 * x)

interact(f, x=widgets.IntSlider(min=0, max=10));

### Un gráfico interactivo (slide 57)

In [ ]:
def graficar(columna):
    sns.relplot(data=propinas_df, x="total_bill", y="tip", hue=columna)

interact(graficar,
         columna=widgets.Dropdown(options=["smoker", "sex", "time"]));